<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">《从零构建推理模型》</a>一书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 8 章：练习解答

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 8.1：训练集和验证集的长度

- 以下单元重新实现第 8.4.3 节末尾的内容，并使用 `reasoning_from_scratch.ch08` 中可复用的辅助函数计算训练集和验证集的长度统计信息
- 如果仍打开着第 8 章 notebook，也可以忽略下面的代码，只需执行：

```python
print("Training stats:")
compute_length(train_examples)

print("Validation stats:")
compute_length(val_examples)
```


In [ ]:
import random

from reasoning_from_scratch.ch08 import (
    build_examples,
    compute_length,
    filter_examples_by_max_len,
    load_distill_data,
    load_reasoning_tokenizer,
)

In [ ]:
math_train = load_distill_data(partition="deepseek-r1-math-train")
tokenizer = load_reasoning_tokenizer()

examples, skipped = build_examples(math_train, tokenizer)
print("Number of examples:", len(examples))
print("Number of skipped examples:", skipped)

compute_length(examples)

filtered_examples = filter_examples_by_max_len(examples, max_len=2048)
compute_length(filtered_examples)

In [ ]:
rng = random.Random(123)
rng.shuffle(filtered_examples)

train_examples = filtered_examples[25:]
val_examples = filtered_examples[:25]

In [ ]:
print("Training stats:")
compute_length(train_examples)

In [ ]:
print("Validation stats:")
compute_length(val_examples)

- 可以看到，平均词元长度（1180 与 1106）相当接近，两个数据集应当比较均衡
- 作为补充，还可以绘制直方图来可视化分布


In [ ]:
import matplotlib.pyplot as plt

train_lengths = [len(ex["token_ids"]) for ex in train_examples]
val_lengths = [len(ex["token_ids"]) for ex in val_examples]

# 验证集小得多，因此对计数进行归一化
bins = range(0, max(train_lengths + val_lengths) + 64, 64)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(train_lengths, bins=bins, density=True, alpha=0.6, label="Train")
ax.hist(val_lengths, bins=bins, density=True, alpha=0.6, label="Validation")
ax.set_xlabel("Token length")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig("1.pdf")
plt.show()

- 验证样本少得多，因此验证集直方图看起来略显参差；不过可以看到，它仍覆盖了良好的分布范围。

&nbsp;
## 练习 8.2：不使用 `<think>` 词元进行蒸馏

- 要复现不使用 `<think></think>` 词元的运行，请执行：

```bash
uv run distill.py \
--data_path deepseek-r1-math-train.json \
--validation_size 25 \
--epochs 3 \
--lr 1e-5 \
--max_seq_len 2048 \
--grad_clip 1.0
```


- 随后在评估时使用 `base` 模型，而不是 `reasoning` 模型：

```bash
uv run evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--max_new_tokens 4096 \
--checkpoint_path run_11/checkpoints/distill/qwen3-0.6B-distill-step05746-epoch1.pth
```


- 新结果（不使用 think 词元）显示在前，对应的使用 think 词元结果置于括号中

| 教师数据                          | 轮次 | MATH-500 准确率 | 最终验证损失       |
|-----------------------------------|------:|--------------:|---------------------:|
| 基础模型（第 3 章）               |     - | 15.2%         |              -       |
| 推理模型（第 3 章）               |     - | 48.2%         |              -       |
| DeepSeek R1 蒸馏数据              |     1 | 31.8% (30.6%) | 0.5436   (0.5404)    |
| DeepSeek R1 蒸馏数据              |     2 | 31.8% (32.4%) | 0.5349   (0.5339)    |
| DeepSeek R1 蒸馏数据              |     3 | 30.2% (33.6%) | 0.5343   (0.5306)    |
| Qwen3 235B A22B 蒸馏数据          |     1 | 44.8% (45.0%) | 0.4043   (0.3130)    |
| Qwen3 235B A22B 蒸馏数据          |     2 | 39.4% (43.8%) | 0.3963   (0.3087)    |
| Qwen3 235B A22B 蒸馏数据          |     3 | 39.8% (44.2%) | 0.3948   (0.3078)    |


- 有趣的是，省略 `<think></think>` 词元后，Qwen3 模型的验证损失更低，但这并未转化为更好的建模性能
- 可以看到，省略 `<think></think>` 几乎在所有情况下都会使结果略微变差
